In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import boto3
from botocore.config import Config
from dotenv import load_dotenv
from io import StringIO
import os
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def clean(s):
    return (s or "").strip().replace("\r", "").replace("\n", "")


# Charger variables d'environnement si besoin
load_dotenv()

session = boto3.Session(
    aws_access_key_id=clean(os.getenv("AWS_ACCESS_KEY_ID")),
    aws_secret_access_key=clean(os.getenv("AWS_SECRET_ACCESS_KEY")),
    region_name="eu-west-3",
)
s3 = session.client("s3", config=Config(signature_version="s3v4"))
BUCKET = "mygeodechetbuckets3"


def read_csv_robust(
    body_bytes: bytes, default_sep: str = ",", try_utf8sig_first: bool = True
) -> pd.DataFrame:
    """Lecture robuste : essaie utf-8-sig puis latin-1, set sep, et répare la mojibake si besoin."""
    df = None
    if try_utf8sig_first:
        try:
            df = pd.read_csv(StringIO(body_bytes.decode("utf-8-sig")), sep=default_sep)
        except Exception:
            pass
    if df is None:
        try:
            df = pd.read_csv(StringIO(body_bytes.decode("latin-1")), sep=default_sep)
        except Exception:
            # dernier fallback: utf-8 simple
            df = pd.read_csv(StringIO(body_bytes.decode("utf-8")), sep=default_sep)
    # Nettoyage BOM dans colonnes
    cols = [c.replace("\ufeff", "") for c in df.columns]
    # Répare si mojibake du style DÃ©partement, annÃ©e, ï»¿Code, etc.
    if any(("Ã" in c) or ("ï»¿" in c) or ("Â" in c) for c in cols):
        cols = [
            c.encode("latin-1", "ignore")
            .decode("utf-8", "ignore")
            .replace("\ufeff", "")
            for c in cols
        ]
    df.columns = cols
    return df


# data_wip_v5.csv : UTF-8-SIG, séparateur point-virgule
obj = s3.get_object(Bucket=BUCKET, Key="data_wip_v5.csv")
df = read_csv_robust(obj["Body"].read(), default_sep=";", try_utf8sig_first=True)

In [ ]:
# ===========================================================
# Helpers + préparation (fonctionne avec TON CSV uniquement)
# ===========================================================


def _numify(s: pd.Series) -> pd.Series:
    """Convertit '20 832,6' -> 20832.6 ; gère NBSP/espaces/virgule ; renvoie float (NaN si invalide)."""
    if hasattr(s, "dtype") and s.dtype.kind in "biufc":
        return s
    s = s.astype(str)
    s = s.str.replace("\u00a0", "", regex=False)  # NBSP
    s = s.str.replace(" ", "", regex=False)  # milliers
    s = s.str.replace(",", ".", regex=False)  # décimale
    s = s.str.replace(r"[^0-9\.\-eE+]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")


def _normalize_years(d: pd.DataFrame, years):
    """Retourne (df filtré, liste triée des années conservées)."""
    d = d.copy()
    d["année"] = _numify(d["année"]).astype(int)
    if years is None:
        return d, sorted(d["année"].unique().tolist())
    years = [int(y) for y in years]
    d = d[d["année"].isin(years)]
    return d, sorted(d["année"].unique().tolist())


def _filter_scope(d: pd.DataFrame, level: str, groups):
    """
    Filtre le DataFrame par périmètre.
    level: 'France' | 'Région' | 'Département'
    groups: None, str/int, ou liste (noms régions, noms départements, ou codes dép).
    """
    d = d.copy()
    d["Code_Dpt"] = d["Code_Dpt"].astype(str).str.zfill(2)

    if (level or "").lower() == "france" or groups is None:
        return d

    if isinstance(groups, (str, int, np.integer)):
        groups = [groups]
    groups_norm = {str(g).lower() for g in groups}

    if (level or "").lower().startswith("r"):  # Région
        return d[d["Région"].str.lower().isin(groups_norm)]
    else:  # Département (accepte noms OU codes)
        mask = d["Département"].str.lower().isin(groups_norm) | d[
            "Code_Dpt"
        ].str.lower().isin(groups_norm)
        return d[mask]


def _mask_dept(d: pd.DataFrame, dept):
    """dept = nom ('Paris') ou code ('75'/75)"""
    if isinstance(dept, (int, np.integer)) or (
        isinstance(dept, str) and dept.isdigit()
    ):
        return d["Code_Dpt"].astype(str).str.zfill(2) == str(dept).zfill(2)
    return d["Département"].str.lower() == str(dept).lower()


# Colonnes des 5 types (selon présence dans TON CSV)
WASTE_COLS = [
    c
    for c in [
        "Déblais_gravats",
        "Déchets_verts",
        "Encombrants",
        "Matériaux_recyclables",
        "Total_autres_dechets",
    ]
    if c in df.columns
]

# S'assurer que les colonnes numériques clés sont bien numériques (idempotent)
for c in [
    c
    for c in [
        "année",
        "pop_globale",
        "pop_globale_n-2",
        "densité",
        "densité_n-2",
        "tonnage_dechet_produit",
        "tonnage_dechet_produit_n-2",
        *WASTE_COLS,
    ]
    if c in df.columns
]:
    df[c] = _numify(df[c])


# ===========================================================
# 1) Tonnage total par année + Évolution (%) — France / Région(s) / Département(s)
# ===========================================================
def plot_tonnage_year_with_growth(
    df_,
    *,
    level="France",
    groups=None,
    years=None,
    title="Tonnage total par année + Évolution (%)",
):
    """
    level: 'France' | 'Région' | 'Département'
    groups: None (tout), ou str/list de régions/départements/codes dép
    years: None (toutes) ou liste d'années
    """
    d = _filter_scope(df_, level, groups)
    if d.empty:
        print("Aucune donnée après filtrage.")
        return
    d, years = _normalize_years(d, years)
    d["tonnage_dechet_produit"] = _numify(d["tonnage_dechet_produit"])

    yearly = (
        d.groupby("année", as_index=False)["tonnage_dechet_produit"]
        .sum()
        .sort_values("année")
    )
    yearly["evol_%"] = yearly["tonnage_dechet_produit"].pct_change() * 100

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(
            x=yearly["année"],
            y=yearly["tonnage_dechet_produit"],
            name="Tonnage Total (tonnes)",
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["année"],
            y=yearly["evol_%"],
            mode="lines+markers",
            name="Évolution (%)",
        ),
        secondary_y=True,
    )
    fig.update_layout(title=title, legend_title_text=None)
    fig.update_xaxes(title_text="Année", dtick=1)
    fig.update_yaxes(title_text="Tonnage (tonnes)", secondary_y=False)
    fig.update_yaxes(title_text="Évolution (%)", secondary_y=True)
    fig.show()


# ===========================================================
# 2) Bar animée — Population par année et par Région OU Département
# ===========================================================
def plot_population_by_year_group(
    df_, *, group="Région", groups=None, years=None, title="Population par année"
):
    """
    group: 'Région' ou 'Département'
    groups: None (tout) ou liste/str de régions/départements/codes dép
    years: None (toutes) ou liste d'années
    """
    d = _filter_scope(df_, group, groups)  # on réutilise le même helper
    if d.empty:
        print("Aucune donnée après filtrage.")
        return
    d, years = _normalize_years(d, years)
    gcol = "Région" if group.lower().startswith("r") else "Département"
    grp = d.groupby([gcol, "année"], as_index=False)["pop_globale"].sum()

    fig = px.bar(
        grp,
        x=gcol,
        y="pop_globale",
        animation_frame="année",
        color=gcol,
        title=f"{title} et par {gcol.lower()}",
        labels={"pop_globale": "Population"},
    )
    fig.update_layout(xaxis_tickangle=-45, legend_title_text=None)
    fig.show()


# ===========================================================
# 3) Bar empilée animée — Tonnage par année et par Région OU Département (5 types)
# ===========================================================
def plot_total_tonnage_by_year_group_types(
    df_,
    *,
    group="Région",
    groups=None,
    years=None,
    title="Tonnage total par année (5 types)",
):
    if not WASTE_COLS:
        print("Colonnes déchets absentes.")
        return
    d = _filter_scope(df_, group, groups)
    if d.empty:
        print("Aucune donnée après filtrage.")
        return
    d, years = _normalize_years(d, years)
    gcol = "Région" if group.lower().startswith("r") else "Département"

    agg = d.groupby([gcol, "année"], as_index=False)[WASTE_COLS].sum()
    dm = agg.melt(
        id_vars=[gcol, "année"],
        value_vars=WASTE_COLS,
        var_name="Type",
        value_name="Tonnage",
    )

    fig = px.bar(
        dm,
        x=gcol,
        y="Tonnage",
        color="Type",
        animation_frame="année",
        title=f"{title} et par {gcol.lower()}",
    )
    fig.update_layout(barmode="stack", xaxis_tickangle=-45, legend_title_text=None)
    fig.show()


# ===========================================================
# 4) Bar — Population cumulée par Région OU Département (toutes années)
# ===========================================================
def plot_population_cumulated_group(
    df_, *, group="Région", groups=None, title="Population cumulée (toutes années)"
):
    d = _filter_scope(df_, group, groups)
    if d.empty:
        print("Aucune donnée après filtrage.")
        return
    gcol = "Région" if group.lower().startswith("r") else "Département"
    grp = d.groupby(gcol, as_index=False)["pop_globale"].sum()

    fig = px.bar(
        grp.sort_values("pop_globale", ascending=False),
        x=gcol,
        y="pop_globale",
        title=f"{title} par {gcol.lower()}",
        labels={"pop_globale": "Population totale"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


# ===========================================================
# 5) Bar — Tonnage total par type (5 types) avec filtres (facultatif)
# ===========================================================
def plot_total_tonnage_by_type(
    df_,
    *,
    level="France",
    groups=None,
    years=None,
    title="Tonnage total par type de déchet",
):
    if not WASTE_COLS:
        print("Colonnes déchets absentes.")
        return
    d = _filter_scope(df_, level, groups)
    if d.empty:
        print("Aucune donnée après filtrage.")
        return
    d, _ = _normalize_years(d, years)
    totals = d[WASTE_COLS].sum().reset_index()
    totals.columns = ["Type", "Tonnage"]
    fig = px.bar(
        totals.sort_values("Tonnage", ascending=False),
        x="Type",
        y="Tonnage",
        title=title,
    )
    fig.update_layout(xaxis_tickangle=-20)
    fig.show()


# ===========================================================
# 6) (Option) Carte animée — type de déchet dominant par département (par année)
#     -> Fournir un geojson (dict) ou une URL; sinon ignore cette fonction.
# ===========================================================


def plot_map_dominant_waste_type(
    df_,
    geojson,
    *,
    years=None,
    featureidkey="properties.code",
    dept_key_col="Code_Dpt",
    title="Type de déchet dominant par département (animation par année)",
):
    """
    geojson : dict déjà chargé OU URL vers un GeoJSON des départements.
              Le champ `featureidkey` doit contenir un code qui matche df_[dept_key_col] (ex: '01', '59', ...).
    years   : None -> toutes les années ; sinon liste d'années.
    """
    if isinstance(geojson, str):
        try:
            geojson = requests.get(geojson).json()
        except Exception as e:
            print("Impossible de charger le GeoJSON depuis l’URL :", e)
            return

    d = df_.copy()
    d["année"] = _numify(d["année"]).astype(int)
    d[dept_key_col] = d[dept_key_col].astype(str).str.zfill(2)
    if years is not None:
        years = [int(y) for y in years]
        d = d[d["année"].isin(years)]

    keep = [dept_key_col, "Département", "année"] + WASTE_COLS
    missing = [c for c in keep if c not in d.columns]
    if missing:
        print("Colonnes manquantes :", missing)
        return

    d = d[keep].copy()
    d["Type_dominant"] = d[WASTE_COLS].idxmax(axis=1)
    d = d.sort_values(WASTE_COLS, ascending=False).drop_duplicates(
        subset=[dept_key_col, "année"]
    )

    fig = px.choropleth(
        d,
        geojson=geojson,
        locations=dept_key_col,
        color="Type_dominant",
        animation_frame="année",
        featureidkey=featureidkey,
        hover_name="Département",
        title=title,
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.show()

In [ ]:
# 1) Tonnage total + Évolution (%) — FRANCE
plot_tonnage_year_with_growth(df, level="France")

#    … pour une région :
plot_tonnage_year_with_growth(df, level="Région", groups="Île-de-France")
#    … pour plusieurs départements (noms ou codes) :
plot_tonnage_year_with_growth(
    df,
    level="Département",
    groups=["Nord", "59", "75"],
    years=[2009, 2011, 2013, 2015, 2017, 2019, 2021],
)

In [ ]:
# 2) Population par année (animé) — au choix Région ou Département
plot_population_by_year_group(df, group="Région")
plot_population_by_year_group(
    df, group="Département", groups=["Nord", "Pas-de-Calais", "Somme"]
)

In [ ]:
# 3) Tonnage par année (empilé, 5 types) — au choix Région ou Département
plot_total_tonnage_by_year_group_types(df, group="Région")
plot_total_tonnage_by_year_group_types(
    df, group="Département", groups=["75", "92", "93"], years=[2009, 2013, 2017, 2021]
)

In [ ]:
# 4) Population cumulée — au choix Région ou Département
plot_population_cumulated_group(df, group="Région")
plot_population_cumulated_group(
    df, group="Département", groups=["Nord", "Pas-de-Calais", "Somme"]
)

In [ ]:
# 5) Tonnage total par type (5 types) — filtrable
plot_total_tonnage_by_type(df, level="France")
plot_total_tonnage_by_type(df, level="Région", groups="Île-de-France")
plot_total_tonnage_by_type(df, level="Département", groups=["75", "92", "93"])

In [ ]:
# 6) Carte animée (si tu as un GeoJSON dispo)
plot_map_dominant_waste_type(
    df, "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"
)